In [20]:
from tools import ECOHOME_TOOLS

In [15]:
from pathlib import Path
import json
import sys
from datetime import datetime

import pandas as pd

# Make sure the project root is available
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name != "ecohome_solution":
    PROJECT_ROOT = Path.cwd()

sys.path.insert(0, str(PROJECT_ROOT))

from agent import EcoHomeAgent

print("EcoHome evaluation notebook loaded")
print("Project root:", PROJECT_ROOT)

EcoHome evaluation notebook loaded
Project root: c:\Users\praveen.vanaparthy\Downloads\New folder\ecohome_solution


In [16]:
agent = EcoHomeAgent()
print(type(agent))
print(agent.__dict__.keys())

<class 'agent.EcoHomeAgent'>
dict_keys(['instructions', 'model_name', 'llm', 'tool_llm', 'tool_node', 'graph'])


In [17]:
ECOHOME_SYSTEM_PROMPT = """
You are EcoHome Energy Advisor, an AI-powered household energy optimization assistant.

ROLE:
Help homeowners understand electricity usage and provide practical,
data-driven recommendations to reduce energy consumption and electricity cost.

YOUR PROCESS:
1. Understand the user's question and household context.
2. Retrieve relevant energy-efficiency knowledge using RAG when appropriate.
3. Check historical energy usage when device usage or consumption patterns are relevant.
4. Check electricity prices when time-of-use or peak/off-peak decisions matter.
5. Check weather when weather conditions can affect HVAC or solar recommendations.
6. Check solar generation when solar usage or generation is relevant.
7. Calculate potential savings when enough numerical information is available.
8. Combine the available evidence before making a recommendation.
9. Clearly explain the recommendation and the reasoning behind it.
10. Never invent energy measurements, prices, weather information, or savings.

CAPABILITIES:
- Energy usage analysis
- Device-level energy analysis
- Solar generation analysis
- Weather-aware recommendations
- Electricity price optimization
- Energy-efficiency knowledge retrieval
- Savings estimation
- EV charging optimization
- HVAC optimization
- Smart-home energy management
- Energy-storage recommendations

RECOMMENDATION GUIDELINES:
- Prefer practical actions that a homeowner can implement.
- Prioritize actions with measurable cost or energy benefits.
- Use historical data when available.
- Consider peak and off-peak electricity pricing.
- Consider solar generation when available.
- Explain assumptions and limitations.
- Identify simulated/mock external data when applicable.
- Do not claim that an action will save a specific amount unless the available
  data supports the calculation.

EXAMPLE QUESTIONS:
1. How can I reduce my electricity cost during peak hours?
2. Which appliance is consuming the most energy?
3. When should I charge my EV?
4. How can I optimize my HVAC usage?
5. How much could I save by shifting usage to off-peak hours?
6. How can I make better use of my solar generation?
7. Should I consider battery storage?
8. What can I do to reduce my household energy consumption?

FINAL RESPONSE:
Provide a concise, clear and actionable recommendation.
Include relevant numerical findings when available.
"""
print("ECOHOME_SYSTEM_PROMPT created")
print("Prompt length:", len(ECOHOME_SYSTEM_PROMPT))

ECOHOME_SYSTEM_PROMPT created
Prompt length: 2362


In [21]:
agent = EcoHomeAgent(
    instructions=ECOHOME_SYSTEM_PROMPT,
    model="command-a-03-2025"
)

print("EcoHome agent initialized successfully")
print("Model:", agent.model_name)
print("Tool count:", len(ECOHOME_TOOLS))
print("Tools:", [tool.name for tool in ECOHOME_TOOLS])

EcoHome agent initialized successfully
Model: command-a-03-2025
Tool count: 7
Tools: ['get_weather_forecast', 'get_electricity_prices', 'get_energy_usage', 'get_solar_generation', 'get_recent_energy_summary', 'search_energy_tips', 'calculate_savings']


In [22]:
test_cases = [
    {
        "id": "TC01",
        "question": "For 2026-10-08, use the electricity pricing data and internal energy-saving tips to identify the peak and off-peak rates and recommend how I should shift flexible household usage.",
        "expected_tools": ["get_electricity_prices", "search_energy_tips"],
        "expected_response": "Use the electricity pricing data for 2026-10-08 to identify lower-cost periods and use internal energy-saving guidance to recommend shifting flexible loads away from peak hours.",
    },
    {
        "id": "TC02",
        "question": "Use my historical energy-usage data to identify which devices consumed the most electricity during the last 7 days, then summarize the finding.",
        "expected_tools": ["get_energy_usage"],
        "expected_response": "Identify the household devices with the highest energy consumption during the last 7 days and summarize their historical usage.",
    },
    {
        "id": "TC03",
        "question": "Use the electricity pricing data and internal energy-saving tips to recommend the best EV charging schedule for the current time-of-use rates. Do not guess the rates.",
        "expected_tools": ["get_electricity_prices", "search_energy_tips"],
        "expected_response": "Use the available electricity prices to recommend lower-cost EV charging periods and support the recommendation with internal energy-saving guidance.",
    },
    {
        "id": "TC04",
        "question": "For the specific date 2026-10-09, call the weather forecast tool and use the returned weather conditions together with internal energy-saving tips to recommend how I should adjust HVAC usage.",
        "expected_tools": ["get_weather_forecast", "search_energy_tips"],
        "expected_response": "Use the weather forecast for 2026-10-09 and internal energy-saving guidance to recommend practical HVAC efficiency actions based on the returned conditions.",
    },
    {
        "id": "TC05",
        "question": "Use my historical solar-generation data to report how much solar energy has been generated during the last 7 days.",
        "expected_tools": ["get_solar_generation"],
        "expected_response": "Report the recent solar-generation amount from the available historical data and summarize the result.",
    },
    {
        "id": "TC06",
        "question": "Use my recent solar-generation data and internal energy-saving tips to recommend how I can shift suitable household loads into periods when solar generation is available.",
        "expected_tools": ["get_solar_generation", "search_energy_tips"],
        "expected_response": "Use recent solar-generation data and internal energy-efficiency guidance to recommend shifting suitable loads into solar-production periods.",
    },
    {
        "id": "TC07",
        "question": "For the specific date 2026-10-10, use the weather forecast tool and internal energy-saving tips to recommend ways to reduce cooling-related electricity consumption.",
        "expected_tools": ["get_weather_forecast", "search_energy_tips"],
        "expected_response": "Use the returned weather conditions and internal energy-saving guidance to recommend practical ways to reduce cooling demand and electricity consumption.",
    },
    {
        "id": "TC08",
        "question": "Use the calculate_savings tool with exactly 10 kWh, a current peak rate of $0.30/kWh, and an optimized off-peak rate of $0.15/kWh. Tell me the current cost, optimized cost, dollar savings, and percentage reduction.",
        "expected_tools": ["calculate_savings"],
        "expected_response": "Calculate the cost and savings for shifting 10 kWh from $0.30/kWh to $0.15/kWh. The expected savings are $1.50, which is a 50% reduction; current cost is $3.00 and optimized cost is $1.50.",
    },
    {
        "id": "TC09",
        "question": "Use the recent energy summary, solar-generation history, and internal energy-saving tips to evaluate whether battery storage could improve solar self-consumption and reduce grid electricity use.",
        "expected_tools": ["get_recent_energy_summary", "get_solar_generation", "search_energy_tips"],
        "expected_response": "Use recent household energy and solar-generation evidence plus internal guidance to explain whether battery storage could improve solar self-consumption and reduce grid usage, while noting any limitations in the available data.",
    },
    {
        "id": "TC10",
        "question": "Use the recent energy summary and internal energy-saving tips to give me practical recommendations for reducing my overall household electricity consumption.",
        "expected_tools": ["get_recent_energy_summary", "search_energy_tips"],
        "expected_response": "Use recent household energy data and internal energy-saving guidance to provide practical, prioritized recommendations for reducing overall electricity consumption.",
    },
    {
        "id": "TC11",
        "question": "Use the internal energy-saving knowledge base to explain how smart-home automation can reduce electricity usage through scheduling, device control, and temperature management.",
        "expected_tools": ["search_energy_tips"],
        "expected_response": "Use internal energy-efficiency knowledge to explain how smart-home automation can reduce unnecessary electricity use through scheduling, automated device control, and temperature management.",
    },
    {
        "id": "TC12",
        "question": "Use my historical energy-usage data, electricity pricing data, and internal energy-saving tips to recommend how I should schedule my pool pump and washing machine to reduce cost and consumption.",
        "expected_tools": ["get_energy_usage", "get_electricity_prices", "search_energy_tips"],
        "expected_response": "Use historical usage, electricity prices, and internal energy-saving guidance to recommend efficient scheduling and settings for the pool pump and washing machine.",
    },
]

print("Total test cases:", len(test_cases))

for test in test_cases:
    print(test["id"], "-", test["question"])
    print("Expected tools:", test["expected_tools"])
    print("Expected response:", test["expected_response"])


Total test cases: 12
TC01 - For 2026-10-08, use the electricity pricing data and internal energy-saving tips to identify the peak and off-peak rates and recommend how I should shift flexible household usage.
Expected tools: ['get_electricity_prices', 'search_energy_tips']
Expected response: Use the electricity pricing data for 2026-10-08 to identify lower-cost periods and use internal energy-saving guidance to recommend shifting flexible loads away from peak hours.
TC02 - Use my historical energy-usage data to identify which devices consumed the most electricity during the last 7 days, then summarize the finding.
Expected tools: ['get_energy_usage']
Expected response: Identify the household devices with the highest energy consumption during the last 7 days and summarize their historical usage.
TC03 - Use the electricity pricing data and internal energy-saving tips to recommend the best EV charging schedule for the current time-of-use rates. Do not guess the rates.
Expected tools: ['get

In [23]:
def extract_tool_calls(messages):
    """Extract actual requested/executed tool names from the agent trajectory."""
    names = []
    for message in messages or []:
        for tool_call in getattr(message, "tool_calls", []) or []:
            if isinstance(tool_call, dict) and tool_call.get("name"):
                names.append(tool_call["name"])
        if message.__class__.__name__ == "ToolMessage":
            name = getattr(message, "name", None)
            if name:
                names.append(name)
    return list(dict.fromkeys(names))


test_results = []

for test in test_cases:
    print(f"\nRunning {test['id']}...")

    result = agent.invoke(test["question"])
    messages = result.get("messages", [])
    actual_tools = extract_tool_calls(messages)

    test_results.append({
        "test_id": test["id"],
        "question": test["question"],
        "response": result["response"],
        "used_tools": actual_tools,
        "messages_list": messages,
        "expected_tools": test["expected_tools"],
        "expected_response": test["expected_response"],
        "success": result["success"],
        "errors": result["errors"],
    })

    print("Success:", result["success"])
    print("Actual tools from trajectory:", actual_tools)

print("\nAll test cases completed.")
print("Total results:", len(test_results))



Running TC01...
Success: True
Actual tools from trajectory: ['get_electricity_prices', 'search_energy_tips']

Running TC02...
Success: True
Actual tools from trajectory: ['get_energy_usage']

Running TC03...
Success: True
Actual tools from trajectory: ['get_electricity_prices', 'search_energy_tips']

Running TC04...
Success: True
Actual tools from trajectory: ['get_weather_forecast', 'search_energy_tips']

Running TC05...
Success: True
Actual tools from trajectory: ['get_solar_generation']

Running TC06...
Success: True
Actual tools from trajectory: ['get_recent_energy_summary', 'search_energy_tips']

Running TC07...
Success: True
Actual tools from trajectory: ['get_weather_forecast', 'search_energy_tips']

Running TC08...
Success: True
Actual tools from trajectory: ['calculate_savings']

Running TC09...
Success: True
Actual tools from trajectory: ['get_recent_energy_summary', 'get_solar_generation', 'search_energy_tips']

Running TC10...
Success: True
Actual tools from trajectory: ['

In [24]:
results_df = pd.DataFrame(test_results)

display(
    results_df[
        ["test_id", "success", "used_tools", "question"]
    ]
)

print("\nSuccessful tests:",
      results_df["success"].sum(),
      "/",
      len(results_df))

ALL_EXPECTED_TOOLS = {
    "get_weather_forecast",
    "get_electricity_prices",
    "get_energy_usage",
    "get_solar_generation",
    "get_recent_energy_summary",
    "search_energy_tips",
    "calculate_savings",
}

ACTUAL_SUITE_TOOLS = set()
for result in test_results:
    ACTUAL_SUITE_TOOLS.update(result["used_tools"])

suite_missing_tools = sorted(ALL_EXPECTED_TOOLS - ACTUAL_SUITE_TOOLS)

print("\nRequired tool capabilities:", len(ALL_EXPECTED_TOOLS))
print("Tool capabilities actually demonstrated:", len(ALL_EXPECTED_TOOLS - set(suite_missing_tools)))
print("Suite tool coverage:", round(
    len(ALL_EXPECTED_TOOLS - set(suite_missing_tools)) / len(ALL_EXPECTED_TOOLS) * 100,
    2,
), "%")
print("Missing suite tools:", suite_missing_tools or "None")


,test_id,success,used_tools,question
0,TC01,True,"[get_electricity_prices, search_energy_tips]","For 2026-10-08, use the electricity pricing da..."
1,TC02,True,[get_energy_usage],Use my historical energy-usage data to identif...
2,TC03,True,"[get_electricity_prices, search_energy_tips]",Use the electricity pricing data and internal ...
3,TC04,True,"[get_weather_forecast, search_energy_tips]","For the specific date 2026-10-09, call the wea..."
4,TC05,True,[get_solar_generation],Use my historical solar-generation data to rep...
5,TC06,True,"[get_recent_energy_summary, search_energy_tips]",Use my recent solar-generation data and intern...
6,TC07,True,"[get_weather_forecast, search_energy_tips]","For the specific date 2026-10-10, use the weat..."
7,TC08,True,[calculate_savings],Use the calculate_savings tool with exactly 10...
8,TC09,True,"[get_recent_energy_summary, get_solar_generati...","Use the recent energy summary, solar-generatio..."
9,TC10,True,"[get_recent_energy_summary, search_energy_tips]",Use the recent energy summary and internal ene...



Successful tests: 12 / 12

Required tool capabilities: 7
Tool capabilities actually demonstrated: 7
Suite tool coverage: 100.0 %
Missing suite tools: None


In [25]:
llm = agent.llm
print("LLM:", type(llm))

LLM: <class 'langchain_cohere.chat_models.ChatCohere'>


In [26]:
def evaluate_response(response, question, expected_response):
    """
    Evaluate an EcoHome agent response using an LLM-as-a-judge.

    The expected_response is treated as the ground truth/reference answer.

    Scores:
        Accuracy
        Relevance
        Completeness
        Usefulness

    Each metric is scored from 1 to 5.
    """

    if not str(response).strip():
        return {
            "Accuracy": 1,
            "Relevance": 1,
            "Completeness": 1,
            "Usefulness": 1,
            "Feedback": "No response was generated.",
        }

    judge_prompt = f"""
You are an expert evaluator for an energy-management AI assistant.

Evaluate the agent response against the user's question and the expected
reference response.

USER QUESTION:
{question}

EXPECTED RESPONSE / GROUND TRUTH:
{expected_response}

AGENT RESPONSE:
{response}

Score the agent response from 1 to 5 for each criterion:

1. Accuracy - Does the response agree with the expected response and avoid
   unsupported or incorrect claims?
2. Relevance - Does it directly answer the user's question?
3. Completeness - Does it cover the important points present in the expected
   response?
4. Usefulness - Does it provide practical and actionable guidance?

Return ONLY valid JSON in exactly this format:

{{
    "Accuracy": 1,
    "Relevance": 1,
    "Completeness": 1,
    "Usefulness": 1,
    "Feedback": "Brief explanation of the scores."
}}
"""

    try:
        judge_response = llm.invoke(judge_prompt)

        import json

        judge_text = str(judge_response.content).strip()

        # Remove markdown JSON fences if the model returns them.
        if judge_text.startswith("```"):
            judge_text = judge_text.replace("```json", "").replace("```", "").strip()

        result = json.loads(judge_text)

        return {
            "Accuracy": max(1, min(5, int(result.get("Accuracy", 1)))),
            "Relevance": max(1, min(5, int(result.get("Relevance", 1)))),
            "Completeness": max(1, min(5, int(result.get("Completeness", 1)))),
            "Usefulness": max(1, min(5, int(result.get("Usefulness", 1)))),
            "Feedback": str(result.get("Feedback", "LLM evaluation completed.")),
        }

    except Exception as e:
        return {
            "Accuracy": 1,
            "Relevance": 1,
            "Completeness": 1,
            "Usefulness": 1,
            "Feedback": f"LLM judge evaluation failed: {e}",
        }


print("evaluate_response() created successfully")


evaluate_response() created successfully


In [27]:
def evaluate_tool_usage(messages_list, expected_tools):
    """
    Evaluate EcoHome agent tool usage.

    The evaluation combines:
    1. Deterministic extraction of actual tool calls from LangChain messages.
    2. Expected-vs-actual comparison (missing and unexpected tools).
    3. LLM-as-a-judge assessment of appropriateness and completeness.

    Scores:
        Tool Appropriateness: 1-5
        Tool Completeness: 1-5
    """

    expected_tools = list(dict.fromkeys(expected_tools or []))

    if not messages_list:
        return {
            "Tool Appropriateness": 1,
            "Tool Completeness": 1,
            "Actual Tools": [],
            "Executed Tools": [],
            "Missing Tools": expected_tools,
            "Unexpected Tools": [],
            "Feedback": "No agent messages were available for evaluation.",
        }

    tool_usage = []

    for message in messages_list:
        if hasattr(message, "tool_calls") and message.tool_calls:
            for tool_call in message.tool_calls:
                if isinstance(tool_call, dict):
                    tool_name = tool_call.get("name")
                    if tool_name:
                        tool_usage.append(tool_name)

        # ToolMessage objects expose the executed tool name through `name`.
        if message.__class__.__name__ == "ToolMessage":
            tool_name = getattr(message, "name", None)
            if tool_name:
                tool_usage.append(tool_name)

    actual_tools = list(dict.fromkeys(tool_usage))
    executed_tools = []
    for message in messages_list:
        if message.__class__.__name__ == "ToolMessage":
            tool_name = getattr(message, "name", None)
            if tool_name:
                executed_tools.append(tool_name)
    executed_tools = list(dict.fromkeys(executed_tools))
    missing_tools = [tool for tool in expected_tools if tool not in actual_tools]
    unexpected_tools = [tool for tool in actual_tools if tool not in expected_tools]

    judge_prompt = f"""
You are an expert evaluator of an AI energy-management agent.

Evaluate whether the tools selected by the agent were appropriate and complete.

EXPECTED TOOLS:
{expected_tools}

ACTUAL TOOLS USED:
{actual_tools}

MISSING EXPECTED TOOLS:
{missing_tools}

UNEXPECTED TOOLS:
{unexpected_tools}

CONVERSATION MESSAGES:
{messages_list}

Evaluate:
1. Tool Appropriateness - Were the tools actually used relevant to the user's question?
2. Tool Completeness - Were the important expected tools used when they were necessary?

Score each criterion from 1 to 5.

Return ONLY valid JSON in exactly this format:
{{
    "Tool Appropriateness": 1,
    "Tool Completeness": 1,
    "Feedback": "Brief explanation of the evaluation."
}}
"""

    try:
        judge_response = llm.invoke(judge_prompt)
        judge_text = str(judge_response.content).strip()

        if judge_text.startswith("```"):
            judge_text = (
                judge_text.replace("```json", "")
                .replace("```", "")
                .strip()
            )

        result = json.loads(judge_text)

        return {
            "Tool Appropriateness": max(
                1, min(5, int(result.get("Tool Appropriateness", 1)))
            ),
            "Tool Completeness": max(
                1, min(5, int(result.get("Tool Completeness", 1)))
            ),
            "Actual Tools": actual_tools,
            "Executed Tools": executed_tools,
            "Missing Tools": missing_tools,
            "Unexpected Tools": unexpected_tools,
            "Feedback": str(result.get("Feedback", "LLM tool evaluation completed.")),
        }

    except Exception as e:
        return {
            "Tool Appropriateness": 1,
            "Tool Completeness": 1,
            "Actual Tools": actual_tools,
            "Executed Tools": executed_tools,
            "Missing Tools": missing_tools,
            "Unexpected Tools": unexpected_tools,
            "Feedback": f"LLM tool judge evaluation failed: {e}",
        }


print("evaluate_tool_usage() created successfully")


evaluate_tool_usage() created successfully


In [28]:
evaluation_results = []

for test in test_results:
    response_eval = evaluate_response(
        test["response"],
        test["question"],
        test["expected_response"]
    )

    tool_eval = evaluate_tool_usage(
        test["messages_list"],
        test["expected_tools"]
    )

    evaluation_results.append({
        "test_id": test["test_id"],
        "question": test["question"],
        "success": test["success"],
        "used_tools": ", ".join(test["used_tools"]),
        "expected_tools": ", ".join(test["expected_tools"]),
        "actual_tools_from_messages": ", ".join(tool_eval["Actual Tools"]),
        "executed_tools": ", ".join(tool_eval["Executed Tools"]),
        "missing_tools": ", ".join(tool_eval["Missing Tools"]),
        "unexpected_tools": ", ".join(tool_eval["Unexpected Tools"]),
        "tool_coverage_percent": round(
            (
                len(set(test["expected_tools"]) - set(tool_eval["Missing Tools"]))
                / len(set(test["expected_tools"])) * 100
            ),
            2
        ) if test["expected_tools"] else 100.0,

        "Accuracy": response_eval["Accuracy"],
        "Relevance": response_eval["Relevance"],
        "Completeness": response_eval["Completeness"],
        "Usefulness": response_eval["Usefulness"],

        "Tool Appropriateness": tool_eval["Tool Appropriateness"],
        "Tool Completeness": tool_eval["Tool Completeness"],

        "Response Feedback": response_eval["Feedback"],
        "Tool Feedback": tool_eval["Feedback"],
    })

evaluation_df = pd.DataFrame(evaluation_results)

print("Evaluation completed.")
print("Evaluated test cases:", len(evaluation_df))

display(
    evaluation_df[
        [
            "test_id",
            "Accuracy",
            "Relevance",
            "Completeness",
            "Usefulness",
            "Tool Appropriateness",
            "Tool Completeness",
            "tool_coverage_percent",
            "missing_tools",
            "executed_tools",
        ]
    ]
)

print(
    "\nAverage expected-tool coverage:",
    round(evaluation_df["tool_coverage_percent"].mean(), 2),
    "%"
)


actual_suite_tools = set()
for tools_text in evaluation_df["executed_tools"]:
    if tools_text:
        actual_suite_tools.update(t.strip() for t in tools_text.split(","))

missing_suite_tools = sorted(ALL_EXPECTED_TOOLS - actual_suite_tools)
suite_tool_coverage = round(
    len(ALL_EXPECTED_TOOLS - set(missing_suite_tools)) / len(ALL_EXPECTED_TOOLS) * 100,
    2,
)

print("\nEnd-to-end executed-tool coverage:", suite_tool_coverage, "%")
print("Missing required tools across the suite:", missing_suite_tools or "None")


Evaluation completed.
Evaluated test cases: 12


,test_id,Accuracy,Relevance,Completeness,Usefulness,Tool Appropriateness,Tool Completeness,tool_coverage_percent,missing_tools,executed_tools
0,TC01,5,5,5,5,5,5,100.0,,"get_electricity_prices, search_energy_tips"
1,TC02,5,5,5,5,5,5,100.0,,get_energy_usage
2,TC03,4,5,4,5,5,5,100.0,,"get_electricity_prices, search_energy_tips"
3,TC04,5,5,5,5,5,5,100.0,,"get_weather_forecast, search_energy_tips"
4,TC05,5,5,5,5,5,5,100.0,,get_solar_generation
5,TC06,5,5,5,5,3,2,50.0,get_solar_generation,"get_recent_energy_summary, search_energy_tips"
6,TC07,5,5,5,5,5,5,100.0,,"get_weather_forecast, search_energy_tips"
7,TC08,5,5,5,5,5,5,100.0,,calculate_savings
8,TC09,5,5,5,5,5,5,100.0,,"get_recent_energy_summary, get_solar_generatio..."
9,TC10,5,5,5,5,5,5,100.0,,"get_recent_energy_summary, search_energy_tips"



Average expected-tool coverage: 95.83 %

End-to-end executed-tool coverage: 100.0 %
Missing required tools across the suite: None


In [29]:
def generate_evaluation_report(evaluation_df):
    """
    Generate an overall EcoHome agent evaluation report using LLM-generated
    strengths, weaknesses, and recommendations.
    """

    response_metrics = [
        "Accuracy",
        "Relevance",
        "Completeness",
        "Usefulness",
    ]

    tool_metrics = [
        "Tool Appropriateness",
        "Tool Completeness",
    ]

    response_scores = {
        metric: round(evaluation_df[metric].mean(), 2)
        for metric in response_metrics
    }

    tool_scores = {
        metric: round(evaluation_df[metric].mean(), 2)
        for metric in tool_metrics
    }

    tool_coverage_percent = round(
        evaluation_df["tool_coverage_percent"].mean(), 2
    )

    suite_executed_tools = set()
    for tools_text in evaluation_df["executed_tools"]:
        if tools_text:
            suite_executed_tools.update(t.strip() for t in tools_text.split(","))

    suite_missing_tools = sorted(ALL_EXPECTED_TOOLS - suite_executed_tools)
    suite_tool_coverage = round(
        len(ALL_EXPECTED_TOOLS - set(suite_missing_tools)) / len(ALL_EXPECTED_TOOLS) * 100,
        2,
    )

    overall_score = round(
        sum(response_scores.values()) +
        sum(tool_scores.values()),
        2
    )

    max_score = 5 * (
        len(response_metrics) + len(tool_metrics)
    )

    overall_percentage = round(
        (overall_score / max_score) * 100,
        2
    )

    successful_tests = int(evaluation_df["success"].sum())
    total_tests = len(evaluation_df)

    # Prepare evaluation context for the LLM
    feedback_context = evaluation_df[
        [
            "test_id",
            "question",
            "Accuracy",
            "Relevance",
            "Completeness",
            "Usefulness",
            "Tool Appropriateness",
            "Tool Completeness",
            "tool_coverage_percent",
            "missing_tools",
            "Response Feedback",
            "Tool Feedback",
        ]
    ].to_string(index=False)

    judge_prompt = f"""
You are an expert evaluator reviewing an AI-powered home energy assistant.

Analyze the evaluation results below.

OVERALL RESPONSE SCORES:
{response_scores}

OVERALL TOOL SCORES:
{tool_scores}

AVERAGE EXPECTED-TOOL COVERAGE:
{tool_coverage_percent}%

SUITE-LEVEL EXECUTED TOOL COVERAGE:
{suite_tool_coverage}%

MISSING REQUIRED TOOLS ACROSS THE SUITE:
{suite_missing_tools}

OVERALL PERCENTAGE:
{overall_percentage}%

DETAILED TEST RESULTS:
{feedback_context}

Generate:
1. Strengths - the most important areas where the agent performed well.
2. Weaknesses - the most important areas that need improvement.
3. Recommendations - specific, practical improvements based on the evaluation.

Do not invent results that are not supported by the evaluation data.

Return ONLY valid JSON in exactly this format:

{{
    "strengths": [
        "strength 1",
        "strength 2"
    ],
    "weaknesses": [
        "weakness 1",
        "weakness 2"
    ],
    "recommendations": [
        "recommendation 1",
        "recommendation 2",
        "recommendation 3"
    ]
}}
"""

    try:
        judge_response = llm.invoke(judge_prompt)

        import json

        judge_text = str(judge_response.content).strip()

        if judge_text.startswith("```"):
            judge_text = (
                judge_text
                .replace("```json", "")
                .replace("```", "")
                .strip()
            )

        llm_analysis = json.loads(judge_text)

        strengths = llm_analysis.get("strengths", [])
        weaknesses = llm_analysis.get("weaknesses", [])
        recommendations = llm_analysis.get("recommendations", [])

    except Exception as e:
        strengths = ["Evaluation workflow completed successfully."]
        weaknesses = ["LLM analysis could not be generated."]
        recommendations = [
            f"Review the evaluation results manually. LLM analysis error: {e}"
        ]

    report = {
        "total_tests": total_tests,
        "successful_tests": successful_tests,
        "success_rate_percent": round(
            successful_tests / total_tests * 100,
            2
        ) if total_tests else 0,

        "response_scores": response_scores,
        "tool_scores": tool_scores,
        "tool_coverage_percent": tool_coverage_percent,
        "suite_tool_coverage_percent": suite_tool_coverage,
        "suite_missing_tools": suite_missing_tools,

        "overall_score": overall_score,
        "maximum_score": max_score,
        "overall_percentage": overall_percentage,

        "strengths": strengths,
        "weaknesses": weaknesses,
        "recommendations": recommendations,
    }

    return report


def display_evaluation_report(evaluation_report):
    """
    Display the EcoHome evaluation report.
    """

    print("Evaluation Report")
    print("=" * 60)

    print(
        f"Tests: {evaluation_report['successful_tests']}/"
        f"{evaluation_report['total_tests']}"
    )

    print(
        f"Success Rate: "
        f"{evaluation_report['success_rate_percent']}%"
    )

    print(
        f"Overall Score: "
        f"{evaluation_report['overall_score']}/"
        f"{evaluation_report['maximum_score']}"
    )

    print(
        f"Overall Percentage: "
        f"{evaluation_report['overall_percentage']}%"
    )

    print("\nResponse Scores:")
    for metric, score in evaluation_report["response_scores"].items():
        print(f"- {metric}: {score}/5")

    print("\nTool Scores:")
    for metric, score in evaluation_report["tool_scores"].items():
        print(f"- {metric}: {score}/5")

    print(
        f"\nAverage Expected-Tool Coverage: "
        f"{evaluation_report['tool_coverage_percent']}%"
    )

    print(
        f"Suite-level executed-tool coverage: "
        f"{evaluation_report['suite_tool_coverage_percent']}%"
    )

    print(
        "Missing required tools across suite:",
        evaluation_report["suite_missing_tools"] or "None"
    )

    print("\nStrengths:")
    for item in evaluation_report["strengths"]:
        print("-", item)

    print("\nWeaknesses:")
    for item in evaluation_report["weaknesses"]:
        print("-", item)

    print("\nRecommendations:")
    for item in evaluation_report["recommendations"]:
        print("-", item)


evaluation_report = generate_evaluation_report(evaluation_df)

display_evaluation_report(evaluation_report)


Evaluation Report
Tests: 12/12
Success Rate: 100.0%
Overall Score: 29.42/30
Overall Percentage: 98.07%

Response Scores:
- Accuracy: 4.92/5
- Relevance: 5.0/5
- Completeness: 4.92/5
- Usefulness: 5.0/5

Tool Scores:
- Tool Appropriateness: 4.83/5
- Tool Completeness: 4.75/5

Average Expected-Tool Coverage: 95.83%
Suite-level executed-tool coverage: 100.0%
Missing required tools across suite: None

Strengths:
- High accuracy, relevance, completeness, and usefulness in responses across all test cases, as evidenced by consistently high scores in these categories.
- Excellent tool appropriateness and completeness, with 100% suite-level executed tool coverage and 95.83% average expected-tool coverage, indicating robust tool selection and usage.
- Strong ability to provide practical, actionable advice tailored to user queries, as demonstrated in test cases like TC01, TC02, and TC04.
- Effective integration of multiple data sources (e.g., weather forecasts, energy usage, pricing data) to deli

In [30]:
print(test_results[0].keys())
print("\nFirst test messages:")
print(test_results[0]["messages_list"])

dict_keys(['test_id', 'question', 'response', 'used_tools', 'messages_list', 'expected_tools', 'expected_response', 'success', 'errors'])

First test messages:
[SystemMessage(content="\nYou are EcoHome Energy Advisor, an AI-powered household energy optimization assistant.\n\nROLE:\nHelp homeowners understand electricity usage and provide practical,\ndata-driven recommendations to reduce energy consumption and electricity cost.\n\nYOUR PROCESS:\n1. Understand the user's question and household context.\n2. Retrieve relevant energy-efficiency knowledge using RAG when appropriate.\n3. Check historical energy usage when device usage or consumption patterns are relevant.\n4. Check electricity prices when time-of-use or peak/off-peak decisions matter.\n5. Check weather when weather conditions can affect HVAC or solar recommendations.\n6. Check solar generation when solar usage or generation is relevant.\n7. Calculate potential savings when enough numerical information is available.\n8. Combin

In [31]:
# Save detailed test results
results_df.to_csv(
    PROJECT_ROOT / "evaluation_test_results.csv",
    index=False
)

# Save evaluation scores
evaluation_df.to_csv(
    PROJECT_ROOT / "evaluation_scores.csv",
    index=False
)

# Save the complete evaluation report
with open(
    PROJECT_ROOT / "evaluation_report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        evaluation_report,
        f,
        indent=2
    )

print("Evaluation artifacts saved successfully.")

print("\nCreated:")
print("- evaluation_test_results.csv")
print("- evaluation_scores.csv")
print("- evaluation_report.json")
print("- Required suite tools:", sorted(ALL_EXPECTED_TOOLS))

Evaluation artifacts saved successfully.

Created:
- evaluation_test_results.csv
- evaluation_scores.csv
- evaluation_report.json
- Required suite tools: ['calculate_savings', 'get_electricity_prices', 'get_energy_usage', 'get_recent_energy_summary', 'get_solar_generation', 'get_weather_forecast', 'search_energy_tips']
